# Predicting Irrigation Needs: Exploratory Data Analysis and CatBoost Modeling

In this notebook we explore the **Playground Series Season 6 Episode 4 (PS S6E4)** dataset, which contains soil, crop and climate features for farmland. The goal is to predict whether a field needs irrigation based on these features. By understanding the patterns in the data and building a robust model, we aim to help farmers and agronomists make informed irrigation decisions.

We'll work through the following steps:

- **Load and inspect the dataset** – load the train and test sets and preview the structure.
- **Explore the target distribution** – check for class imbalance and understand how many fields require irrigation.
- **Examine categorical features** – visualize the distribution of soil types, crop types and other categories.
- **Analyze numerical features** – understand the relationships between continuous variables and irrigation.
- **Feature engineering** – create new features to capture interactions between soil and crop types.
- **Modeling with CatBoost** – train and tune a CatBoost classifier using cross‑validation and Optuna.

  **Evaluate and conclude** – summarize model performance and insights for irrigation decisions.



Use the table of contents below to jump to any section:

- [Import libraries](#import-libraries)
- [Load dataset](#load-dataset)
- [Target distribution](#target-distribution)
- [Categorical feature analysis](#categorical-feature-analysis)
- [Numerical feature analysis](#numerical-feature-analysis)
- [Feature engineering](#feature-engineering)
- [Feature correlation](#feature-correlation)
- [Modeling and hyperparameter tuning](#modeling-and-hyperparameter-tuning)
- [Results and conclusion](#results-and-conclusion)


## Import libraries

We'll start by importing the essential libraries for data processing, visualization, modeling, hyperparameter tuning and evaluation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import optuna

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score

## 📂 Load dataset


We load the dataset and perform a quick inspection.

In [1]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.head()

NameError: name 'pd' is not defined

## 📊 Target Distribution

Understanding class balance is critical for choosing evaluation metrics.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=train["Irrigation_Need"])
plt.title("Target Distribution")
plt.show()

## 🧹 Data Preprocessing

- Remove ID column
- Separate features and target
- Encode target labels

In [ ]:
# Store test IDs
test_ids = test['id']

# Drop ID columns
train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

# Split features and target
X = train.drop(columns=['Irrigation_Need'])
y_raw = train['Irrigation_Need']

# Encode target
le = LabelEncoder()
y = le.fit_transform(y_raw)

print("Classes:", le.classes_)

## 🧩 Categorical Features

CatBoost handles categorical features internally.

In [ ]:
cat_features = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region"
]

# Sanity check
missing_cols = [col for col in cat_features if col not in X.columns]
print("Missing categorical columns:", missing_cols)

## 📊 Categorical Feature Distribution

In [ ]:
for col in cat_features[:4]:  # limit to avoid clutter
    plt.figure(figsize=(6,4))
    sns.countplot(data=train, x=col, hue="Irrigation_Need")
    plt.xticks(rotation=45)
    plt.title(f"{col} vs Target")
    plt.show()

## 📈 Numerical Features vs Target

In [ ]:
num_cols = X.select_dtypes(include=np.number).columns

for col in num_cols[:5]:  # limit
    plt.figure(figsize=(6,4))
    sns.boxplot(x=train["Irrigation_Need"], y=train[col])
    plt.title(f"{col} distribution by target")
    plt.show()

## ⚙️ Basic Feature Engineering

We create simple interaction features to improve model performance.

In [ ]:
# Example interactions (cheap but effective)
X["Soil_Crop"] = X["Soil_Type"].astype(str) + "_" + X["Crop_Type"].astype(str)
test["Soil_Crop"] = test["Soil_Type"].astype(str) + "_" + test["Crop_Type"].astype(str)

cat_features.append("Soil_Crop")

## 🔗 Feature Correlation

In [ ]:
plt.figure(figsize=(10,8))
corr = X.select_dtypes(include=np.number).corr()

sns.heatmap(corr, cmap="coolwarm", center=0,annot=True,fmt='0.2f')
plt.title("Correlation Heatmap")
plt.show()

## 🎯 Hyperparameter Optimization

We use Optuna with:
- TPE Sampler (efficient search)
- Median Pruner (early stopping for bad trials)

Evaluation metric: **Balanced Accuracy**

In [ ]:
def objective(trial):

    params = {
        # Core
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1",
        "random_seed": 42,
        "verbose": 0,
        "task_type": "GPU",

        # Tree structure
        "depth": trial.suggest_int("depth", 4, 10),

        # Learning
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),

        # Regularization
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),

        # Randomness
        "random_strength": trial.suggest_float("random_strength", 1e-2, 5.0, log=True),

        # Class imbalance
        "auto_class_weights": trial.suggest_categorical(
            "auto_class_weights", ["Balanced", None]
        ),

        # Bootstrap
        "bootstrap_type": trial.suggest_categorical(
            "bootstrap_type", ["Bayesian", "Bernoulli"]
        ),
    }

    # Conditional params
    if params["bootstrap_type"] == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0, 5)
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.6, 1.0)

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scores = []
    best_iters = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_val = y[train_idx], y[valid_idx]

        model = CatBoostClassifier(
            **params,
            iterations=3000,  # large cap
            early_stopping_rounds=100,
            use_best_model=True
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_features,
            verbose=False
        )

        preds = model.predict(X_val).astype(int).flatten()
        score = balanced_accuracy_score(y_val, preds)

        scores.append(score)
        best_iters.append(model.get_best_iteration())

        # pruning
        trial.report(np.mean(scores), step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    trial.set_user_attr("best_iteration", int(np.mean(best_iters)))

    return float(np.mean(scores))

## 🚀 Run Optuna Study

In [ ]:
# Reduce verbosity of Optuna logging to avoid clutter
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42, multivariate=False),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

study.optimize(objective, n_trials=50) 

## 📈 Optuna Optimization Progress

In [ ]:
history = study.trials_dataframe()

plt.figure(figsize=(8,4))
plt.plot(history["value"])
plt.xlabel("Trial")
plt.ylabel("Score")
plt.title("Optimization Progress")
plt.show()

In [ ]:
study.best_params

## 🎯 Hyperparameter Importance

In [ ]:
optuna.visualization.plot_param_importances(study)

In [ ]:
best_params = study.best_trial.params

best_params.update({
    "loss_function": "MultiClass",
    "eval_metric": "TotalF1",
    "random_seed": 42,
    "verbose": 0,
    "task_type": "GPU"
})

best_iteration = study.best_trial.user_attrs["best_iteration"]

print("Best Iteration:", best_iteration)
print("Best Params:", best_params)

## 📊 Optuna Optimization History

In [ ]:
optuna.visualization.plot_optimization_history(study)

## 🔁 Cross Validation + OOF Predictions

We:
- Train using best Optuna parameters
- Use optimal number of iterations
- Generate OOF predictions
- Average test predictions across folds

In [ ]:
n_classes = len(np.unique(y))

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X), dtype=int)
oof_proba = np.zeros((len(X), n_classes))

test_preds_proba = np.zeros((len(test), n_classes))

scores = []

In [ ]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n===== Fold {fold+1} =====")

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    model = CatBoostClassifier(
        **best_params,
        iterations=best_iteration,  # 🔥 KEY FIX
        use_best_model=True
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_features,
        verbose=False
    )

    # --- Validation predictions ---
    val_proba = model.predict_proba(X_val)
    val_preds = np.argmax(val_proba, axis=1)

    oof_preds[val_idx] = val_preds
    oof_proba[val_idx] = val_proba

    # --- Score ---
    score = balanced_accuracy_score(y_val, val_preds)
    scores.append(score)

    print(f"Fold Score: {score:.5f}")

    # --- Test predictions ---
    test_preds_proba += model.predict_proba(test) / kf.n_splits

## 📊 Cross Validation Results

In [ ]:
print("\n===== CV RESULTS =====")
print("Mean Score:", np.mean(scores))
print("Std Dev:", np.std(scores))

## 📋 OOF Classification Report

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y, oof_preds, target_names=le.classes_))

## 📈 OOF Prediction Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(oof_preds, bins=n_classes)
plt.title("OOF Prediction Distribution")
plt.show()

## 🧪 Final Test Predictions

In [ ]:
test_preds = np.argmax(test_preds_proba, axis=1)

## 🏁 Train Final Model on Full Data

In [ ]:
final_model = CatBoostClassifier(
    **best_params,
    iterations=best_iteration  
)

final_model.fit(
    X,
    y,
    cat_features=cat_features,
    verbose=False
)

## ⭐ Feature Importance (CatBoost)

In [ ]:
importances = final_model.get_feature_importance()
feature_names = X.columns

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(data=fi.head(15), x="importance", y="feature")
plt.title("Top 15 Important Features")
plt.show()

## 📤 Generate Predictions and Create Submission

In [ ]:
y_pred = final_model.predict(test).flatten().astype(int)

y_pred_labels = le.inverse_transform(y_pred)

submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": y_pred_labels
})

submission.head()

In [ ]:
submission.to_csv("submission_1.csv", index=False)

##  Results and Conclusion

Our exploratory analysis revealed patterns in the Playground Series PS S6E4 dataset. The target variable (irrigation need) is slightly imbalanced, with more fields not requiring irrigation than those that do. Visualizing the categorical features showed that certain soil types and crop stages are associated with higher irrigation needs, while numeric features such as rainfall and temperature have notable relationships with the target.

After cleaning the data and engineering a few simple features (such as combining soil and crop types), we trained a CatBoost classifier. Hyperparameter tuning with Optuna allowed us to efficiently search the parameter space and find a robust configuration. Cross‑validation demonstrated consistent performance, and the optimized model achieved strong balanced accuracy and F1 scores across classes. CatBoost‑generated feature importances indicated that soil type, crop type and climate variables are among the most influential predictors.

Key takeaways from this analysis:

- **Class distribution:** The dataset contains more instances without irrigation needs, so using balanced accuracy and F1 metrics helps account for class imbalance.
- **Categorical patterns:** Certain combinations of soil and crop types exhibit higher irrigation needs. Exploring these distributions provides actionable insights for agronomists.
- **Numeric relationships:** Features like rainfall and temperature show trends with the target. Understanding these relationships can inform irrigation scheduling.
- **Model performance:** The tuned CatBoost model provides reliable predictions and forms a solid baseline for this problem. Feature importances highlight which variables drive the model's decisions.

### Next steps

- **Try additional models:** Evaluate other tree‑based algorithms (e.g., LightGBM, XGBoost) or ensemble methods to compare performance.
- **Address class imbalance:** Experiment with oversampling, undersampling or class weights to further improve minority‑class performance.
- **Feature engineering:** Create interaction features or domain‑specific variables (e.g., combined climate indices) to capture more nuanced relationships.
- **Model interpretability:** Use SHAP values or partial dependence plots to better understand how features influence predictions.

Thank you for reading! I hope this notebook provides a helpful starting point for analyzing irrigation needs in farmland. Feel free to share feedback or suggestions.
**